In [ ]:
import os
import json
import pandas as pd

# Directory path
directory = "${DATA_DIR}/output_cameo/datacenter_expand_output"

# File name segregation
location_scenarios = ["rural_tech_hub", "industrial_park", "urban_edge"]
operation_models = ["AIML_focused", "standard", "high_redundancy"]
parameter_sets = ["param1", "param2", "param3"]

# Container for all rows
data_rows = []

# Loop through JSON files
for filename in os.listdir(directory):
    if filename.startswith("comprehensive_results_") and filename.endswith(".json"):
        name_part = filename.replace("comprehensive_results_", "").replace(".json", "")

        # Extract location_scenario
        matched_scenario = next(
            (s for s in location_scenarios if name_part.startswith(s)), None
        )
        if not matched_scenario:
            print(f"Skipping unrecognized scenario in file: {filename}")
            continue

        remaining = name_part[
            len(matched_scenario) + 1 :
        ]  # remove scenario + underscore

        # Extract operation_model
        matched_model = next(
            (m for m in operation_models if remaining.startswith(m)), None
        )
        if not matched_model:
            print(f"Skipping unrecognized model in file: {filename}")
            continue

        remaining = remaining[len(matched_model) + 1 :]  # remove model + underscore

        # Extract parameter_set
        matched_param = next((p for p in parameter_sets if remaining == p), None)
        if not matched_param:
            print(f"Skipping unrecognized parameter in file: {filename}")
            continue

        # Load JSON content
        filepath = os.path.join(directory, filename)
        with open(filepath, "r") as f:
            try:
                data = json.load(f)
            except Exception as e:
                print(f"Failed to load {filename}: {e}")
                continue

        optimization_params = data.get("optimization_parameters", {})
        n_contingencies = optimization_params.get("n_contingencies")
        expansion_budget_cap = optimization_params.get("expansion_budget_cap")
        minimum_reliability_index = optimization_params.get("minimum_reliability_index")
        voltage_deviation_limit = optimization_params.get("voltage_deviation_limit")
        pareto_designs = data.get("pareto_front", {}).get("designs", [])
        for design in pareto_designs:
            for line in design.get("selected_lines", []):
                row = {
                    "file_name": filename,
                    "location_scenario": matched_scenario,
                    "operation_model": matched_model,
                    "parameter_set": matched_param,
                    "design_id": design.get("design_id"),
                    "reliability_weight": design.get("reliability_weight"),
                    "cost_weight": design.get("cost_weight"),
                    "reliability_index": design.get("reliability_index"),
                    "expansion_cost": design.get("expansion_cost"),
                    "expected_datacenter_load_shed": design.get(
                        "expected_datacenter_load_shed"
                    ),
                    "expected_total_load_shed": design.get("expected_total_load_shed"),
                    "line_id": line.get("line_id"),
                    "n_contingencies": n_contingencies,
                    "expansion_budget_cap": expansion_budget_cap,
                    "minimum_reliability_index": minimum_reliability_index,
                    "voltage_deviation_limit": voltage_deviation_limit,
                }
                data_rows.append(row)

# Convert to DataFrame and save
df = pd.DataFrame(data_rows)
df.columns = [col.replace("_", " ").title() for col in df.columns]
output_path = os.path.join(directory, "parsed_datacenter_data.csv")
df.to_csv(output_path, index=False)

print(df.head())

                                           File Name Location Scenario  \
0  comprehensive_results_urban_edge_high_redundan...        urban_edge   
1  comprehensive_results_urban_edge_high_redundan...        urban_edge   
2  comprehensive_results_urban_edge_high_redundan...        urban_edge   
3  comprehensive_results_urban_edge_high_redundan...        urban_edge   
4  comprehensive_results_urban_edge_high_redundan...        urban_edge   

   Operation Model Parameter Set  Design Id  Reliability Weight  Cost Weight  \
0  high_redundancy        param3          0            0.100000     0.900000   
1  high_redundancy        param3          1            0.157143     0.842857   
2  high_redundancy        param3          2            0.214286     0.785714   
3  high_redundancy        param3          3            0.271429     0.728571   
4  high_redundancy        param3          4            0.328571     0.671429   

   Reliability Index  Expansion Cost  Expected Datacenter Load Shed  \
0  